In [1]:
'''
Agent with a Memory:

agent that not only remembers but also adapts. We'll challenge the multi_day_trip_agent to re-plan part of its itinerary based on our feedback. 
This is a much more realistic test of conversational AI.
------------------------------------------------------------------------------------------------------------------------------
'''


"\nAgent with a Memory:\n\nagent that not only remembers but also adapts. We'll challenge the multi_day_trip_agent to re-plan part of its itinerary based on our feedback. \nThis is a much more realistic test of conversational AI.\n------------------------------------------------------------------------------------------------------------------------------\n"

In [1]:
import os
import sys
import  json
import asyncio
import random
import string
from uuid import uuid4
from typing import List,Any
import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, Markdown, display

#----------ADK , Agent and Evaluation components Tools Contextimports here------------------

from google.adk.agents import Agent
from google.adk.events import Event
from google.adk.runners import Runner
import google.adk as adk
from google.adk.tools import google_search
from google.adk.sessions import InMemorySessionService, Session
from google.genai import types
from google.genai.types import Content , Part
from dotenv import load_dotenv

import  asyncio
from google.adk.tools import ToolContext
from google.adk.tools.agent_tool  import AgentTool
# Assume 'db_agent' is a pre-defined NL2SQL Agent
# For this example, we'll create placeholder agents

print(" All libraries are imported!")

 All libraries are imported!


In [2]:
load_dotenv()

True

In [3]:
#Runner to Help run the agent: This is a HELPER function
async def run_agent_query(agent:Agent,query:str,session : Session,user_id: str,is_router: bool = False):
    """Initializes a runner and executes a query for a given agent and session."""
    print(f"\n Running query for agent: '{agent.name}' in session: '{session.id}'...")
    runner = Runner(
        agent = agent,
        session_service = session_service,
        app_name = agent.name)
    final_response = ""
    try:
        async for event in runner.run_async(user_id = user_id ,session_id = session.id,new_message = Content(parts=[Part(text = query)],role ="user")):
            if not is_router:
                # Let's see what the agent is thinking! through events 
                print(f"EVENT:{event}")
                if event.is_final_response():
                    final_response = event.content.parts[0].text
    except Exception as e:
        final_response = f"An error occurred: {e}"

    if not is_router:
        print("\n" + "-"*50)
        print("✅ Final Response:")
        display(Markdown(final_response))
        print("-"*50 + "\n")
    return final_response

In [4]:
# --- Initializing Session Service ---
session_service = InMemorySessionService()
my_user_id = "adk_user_001"

In [5]:
db_agent = Agent(
    name = "db_agent",
    model = "gemini-3.5-flash",
    instruction = "You are a database agent. When asked for data, return this mock JSON object: {'status': 'success', 'data': [{'name': 'The Grand Hotel', 'rating': 5, 'reviews': 450}, {'name': 'Seaside Inn', 'rating': 4, 'reviews': 620}]}"
)

In [6]:
#--------------------The Adaptive Planner  Agent ----------------------

def create_multi_day_trip_agent():
    """
    Create the progressive multiagent trip planner agent
    """
    return Agent(
        name = "multi_day_trip_agent",
        model = "gemini-3.5-flash",
        description = "Agent that progressively plans a multi-day trip, remembering previous days and adapting to user feedback.",
        instruction = """
         You are the "Adaptive Trip Planner" 🗺️ - an AI assistant that builds multi-day travel itineraries step-by-step.

        Your Defining Feature:
        You have short-term memory. You MUST refer back to our conversation to understand the trip's context, what has already been planned, and the user's preferences. If the user asks for a change, you must adapt the plan while keeping the unchanged parts consistent.

        Your Mission:
        1.  **Initiate**: Start by asking for the destination, trip duration, and interests.
        2.  **Plan Progressively**: Plan ONLY ONE DAY at a time. After presenting a plan, ask for confirmation.
        3.  **Handle Feedback**: If a user dislikes a suggestion (e.g., "I don't like museums"), acknowledge their feedback, and provide a *new, alternative* suggestion for that time slot that still fits the overall theme.
        4.  **Maintain Context**: For each new day, ensure the activities are unique and build logically on the previous days. Do not suggest the same things repeatedly.
        5.  **Final Output**: Return each day's itinerary in MARKDOWN format.
        """,
        tools = [google_search]
    )

multi_day_trip_agent = create_multi_day_trip_agent()
print(f"🗺️ Agent '{multi_day_trip_agent.name}' is created and ready to plan and adapt!")

NameError: name 'multi_day_agent' is not defined

In [ ]:
#a: Agent WITH Memory (Using a SINGLE Session)
# We will use the exact same trip_session object for the entire conversation. 
#Watch how the agent remembers the context from Turn 1 to correctly handle the requests in Turn 2 and 3.
#---------------------Testing Adaptation and Memory -----------------------------
async def run_adaptive_memory_demonstration():
    print(" AGENT THAT ADAPTS (SAME SESSION) ")
    # Creating ONE session that we will reuse for the whole conversation
    trip_session = await session_service.create_session(
        app_name = multi_day_trip_agent.name,
        user_id = my_user_id
    )
    print(f"Created a single session for our trip: {trip_session.id}")
    #----------------Case 1 : User INITIATES trip conversation--------------------
    query1 = "Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food."
    print(f"\n  User (Case 1): '{query1}'")
    await run_agent_query(multi_day_trip_agent,query1, trip_session , my_user_id)

    #---------------------Case2 : The user gives FEEDBACK and asks for a CHANGE----------------------------
    query2 = "That sounds pretty good, but I'm not a huge fan of castles. Can you replace the morning activity for Day 1 with something else historical?"
    print(f"\n  User (Case 2 - Feedback): '{query2}'")
    await run_agent_query(multi_day_trip_agent, query2, trip_session, my_user_id)
    
    # ---------------- Case 3: The user confirms and asks to continue ----------------
    query3 = "Yes, the new plan for Day 1 is perfect! Please plan Day 2 now, keeping the food theme in mind."
    print(f"\n  User (Case 3 - Confirmation): '{query3}'")
    await run_agent_query(multi_day_trip_agent, query3, trip_session, my_user_id)

await run_adaptive_memory_demonstration()

In [ ]:
#------------------------------3b: Agent WITHOUT Memory (Using SEPARATE Sessions) ----------------------------------
#Now, let's see what happens if we mess up our session management. Here, we'll give the agent a case of amnesia by creating a brand new, separate session for each turn.

#-----------Demonstrating Memory FAILURE --------------------------------
async def run_memory_failure_demonstration():
    print("\n" + "#"*60)
    print(" AGENT WITH AMNESIA (SEPARATE SESSIONS)")
    print("#"*60)

    # --- Turn 1: The user initiates the trip in the FIRST session ---
    query1 = "Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food."
    session_one = await session_service.create_session(
        app_name=multi_day_trip_agent.name,
        user_id=my_user_id
    )
    print(f"\nCreated a session for Turn 1: {session_one.id}")
    print(f" User (Turn 1): '{query1}'")
    await run_agent_query(multi_day_trip_agent, query1, session_one, my_user_id)

    # --- Turn 2: The user asks to continue... but in a completely NEW session ---
    query2 = "Yes, that looks perfect! Please plan Day 2."
    session_two = await session_service.create_session(
        app_name=multi_day_trip_agent.name,
        user_id=my_user_id
    )
    print(f"\nCreated a BRAND NEW session for Turn 2: {session_two.id}")
    print(f" User (Turn 2): '{query2}'")
    await run_agent_query(multi_day_trip_agent, query2, session_two, my_user_id)

await run_memory_failure_demonstration()